## Phase 3 — Fingerprint Scanning

Capture fingerprints (real and fake) using the USB scanner for later comparison.

**Fill in:**
- Type: `real` / `fake`
- Finger: `1–5`
- Hand: `left` / `right`
- UCO: your ID

**Important:** always use the same finger, hand, and UCO.

**Tips:**
- Test different **pressure levels** (light / medium / strong)
- Place your finger in the **center of the scanner**
- Keep the finger **steady** during capture

**Usage:** Run the cell → place finger → **Capture** → repeat → **Stop**

**Saved as:**
{type}\_{finger}\_{hand}\_{UCO}\_{N}.png (e.g. real\_index_left_232886_1.png)

In [3]:
from pathlib import Path
import os
import re
import time
import threading
import subprocess

import cv2
import ipywidgets as widgets
from IPython.display import display


# ===============================================`=============
# SETTINGS
# ============================================================

def find_repo_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd()
    for p in [start, *start.parents]:
        if (p / ".git").exists() or (p / "README.md").exists() or (p / "pyproject.toml").exists():
            return p
    return start


REPO_DIR    = find_repo_root()
INSTALL_DIR = REPO_DIR / "install"
BINARY_PATH = INSTALL_DIR / "ftrScanAPI_Ex"
OUTPUT_FILE = INSTALL_DIR / "frame_Ex.bmp"
SCANS_DIR   = REPO_DIR / "images"
SCANS_DIR.mkdir(parents=True, exist_ok=True)

ENV_VARS = {
    **os.environ,
    "LD_LIBRARY_PATH": f"{INSTALL_DIR}:{os.environ.get('LD_LIBRARY_PATH', '')}"
}

POLL_INTERVAL = 0.25
SCAN_TIMEOUT  = 1.2
PREVIEW_WIDTH  = 320
PREVIEW_HEIGHT = 440

if not INSTALL_DIR.exists():
    raise FileNotFoundError(f"Missing installation directory: {INSTALL_DIR}")
if not BINARY_PATH.exists():
    raise FileNotFoundError(f"Missing scanner binary: {BINARY_PATH}")

print("REPO_DIR   :", REPO_DIR)
print("SCANS_DIR  :", SCANS_DIR)


# ============================================================
# HELPERS
# ============================================================

def read_output_image():
    if not OUTPUT_FILE.exists():
        return None
    return cv2.imread(str(OUTPUT_FILE), cv2.IMREAD_GRAYSCALE)


def encode_png_bytes(img):
    ok, enc = cv2.imencode(".png", img)
    return enc.tobytes() if ok else None


# Maps display label → stored identifier
FINGER_LABELS = {
    "1 - thumb":  "thumb",
    "2 - index":  "index",
    "3 - middle": "middle",
    "4 - ring":   "ring",
    "5 - pinky":  "pinky",
}


def next_capture_path(prefix: str) -> Path:
    """Return next available path: {prefix}_{N}.png"""
    pat = re.compile(rf"^{re.escape(prefix)}_(\d+)\.png$", re.IGNORECASE)
    nums = [int(m.group(1)) for p in SCANS_DIR.iterdir() if (m := pat.match(p.name))]
    n = max(nums) + 1 if nums else 1
    return SCANS_DIR / f"{prefix}_{n}.png"


def save_current_frame(img, prefix: str) -> Path:
    path = next_capture_path(prefix)
    ok = cv2.imwrite(str(path), img)
    if not ok:
        raise IOError(f"Failed to save image to {path}")
    return path


# ============================================================
# STATE
# ============================================================

class ScannerState:
    def __init__(self):
        self.running           = False
        self.thread            = None
        self.last_img          = None
        self.last_output_mtime = None
        self.saved_paths       = []
        self.capture_requested = False
        self.stop_requested    = False


state = ScannerState()


# ============================================================
# WIDGETS
# ============================================================

type_dropdown = widgets.Dropdown(
    options=[None, "real", "fake"],
    value=None,
    description="Type:",
    layout=widgets.Layout(width="160px")
)
finger_dropdown = widgets.Dropdown(
    options=[None] + list(FINGER_LABELS.keys()),
    value=None,
    description="Finger:",
    layout=widgets.Layout(width="220px")
)
hand_dropdown = widgets.Dropdown(
    options=[None, "left", "right"],
    value=None,
    description="Hand:",
    layout=widgets.Layout(width="180px")
)
uco_text = widgets.Text(
    value="",
    placeholder="e.g. 232886",
    description="UCO:",
    layout=widgets.Layout(width="200px")
)
finger_hint = widgets.HTML(
    "<span style='color:gray; font-size:1.1em'>Fingers are numbered from thumb (1) to pinky (5)</span>"
)
main_image = widgets.Image(format="png", width=PREVIEW_WIDTH, height=PREVIEW_HEIGHT)
status_html = widgets.HTML("<b>Status:</b> idle")
saved_html  = widgets.HTML("<b>Saved:</b> none")

capture_button = widgets.Button(description="Capture", button_style="success")
stop_button    = widgets.Button(description="Stop",    button_style="danger")
clear_button   = widgets.Button(description="Clear gallery")

gallery_box = widgets.GridBox(
    [],
    layout=widgets.Layout(grid_template_columns="repeat(4, 150px)", grid_gap="10px")
)

config_row  = widgets.HBox([type_dropdown, finger_dropdown, hand_dropdown, uco_text])
control_row = widgets.HBox([capture_button, stop_button, clear_button])
preview_row = widgets.HBox(
    [main_image, gallery_box],
    layout=widgets.Layout(align_items="flex-start", gap="20px")
)
ui = widgets.VBox([config_row, finger_hint, control_row, status_html, saved_html, preview_row])


# ============================================================
# UI HELPERS
# ============================================================

def refresh_gallery():
    items = []
    for p in reversed(state.saved_paths[-24:]):
        try:
            img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            png = encode_png_bytes(img)
            if png is None:
                continue
            thumb = widgets.Image(value=png, format="png", width=110, height=150)
            label = widgets.HTML(f"<center><code>{p.name}</code></center>")
            items.append(widgets.VBox([thumb, label], layout=widgets.Layout(align_items="center")))
        except Exception as e:
            items.append(widgets.HTML(f"<span style='color:red'>{p.name}: {e}</span>"))
    gallery_box.children = items


def refresh_saved_label():
    if not state.saved_paths:
        saved_html.value = "<b>Saved:</b> none"
        return
    lines = "".join(f"<div><code>{p}</code></div>" for p in state.saved_paths)
    saved_html.value = f"<b>Saved:</b>{lines}"


# ============================================================
# SCANNER LOOP
# ============================================================

def scanner_loop():
    state.running = True
    status_html.value = "<b>Status:</b> running"

    while not state.stop_requested:
        try:
            proc = subprocess.Popen(
                [str(BINARY_PATH)],
                env=ENV_VARS,
                cwd=str(INSTALL_DIR),
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            try:
                proc.wait(timeout=SCAN_TIMEOUT)
            except subprocess.TimeoutExpired:
                proc.kill()

            if OUTPUT_FILE.exists():
                mtime = OUTPUT_FILE.stat().st_mtime
                if state.last_output_mtime != mtime:
                    state.last_output_mtime = mtime
                    img = read_output_image()
                    if img is not None:
                        state.last_img = img
                        png = encode_png_bytes(img)
                        if png is not None:
                            main_image.value = png

            if state.capture_requested and state.last_img is not None:
                state.capture_requested = False
                fp_type      = type_dropdown.value
                finger_label = finger_dropdown.value
                hand         = hand_dropdown.value
                uco          = uco_text.value.strip()
                if fp_type is None:
                    status_html.value = "<b>Status:</b> <span style='color:orange'>Select type (real / fake) before capturing</span>"
                elif finger_label is None:
                    status_html.value = "<b>Status:</b> <span style='color:orange'>Select finger before capturing</span>"
                elif hand is None:
                    status_html.value = "<b>Status:</b> <span style='color:orange'>Select hand (left / right) before capturing</span>"
                elif not uco:
                    status_html.value = "<b>Status:</b> <span style='color:orange'>Enter your UCO before capturing</span>"
                else:
                    prefix = f"{fp_type}_{FINGER_LABELS[finger_label]}_{hand}_{uco}"
                    path = save_current_frame(state.last_img, prefix)
                    state.saved_paths.append(path)
                    saved_html.value = f"<b>Last saved:</b> <code>{path.name}</code>"
                    refresh_gallery()

        except Exception as e:
            status_html.value = f"<b>Status:</b> <span style='color:red'>error: {e}</span>"
            time.sleep(0.5)

        time.sleep(POLL_INTERVAL)

    state.running = False
    status_html.value = "<b>Status:</b> stopped"


# ============================================================
# BUTTON CALLBACKS
# ============================================================

def on_capture_clicked(_):
    state.capture_requested = True
    status_html.value = "<b>Status:</b> capture requested"


def on_stop_clicked(_):
    state.stop_requested = True
    status_html.value = "<b>Status:</b> stopping..."


def on_clear_clicked(_):
    state.saved_paths.clear()
    refresh_gallery()
    refresh_saved_label()


capture_button.on_click(on_capture_clicked)
stop_button.on_click(on_stop_clicked)
clear_button.on_click(on_clear_clicked)

display(ui)

if state.thread is None or not state.thread.is_alive():
    state.stop_requested = False
    state.thread = threading.Thread(target=scanner_loop, daemon=True)
    state.thread.start()

REPO_DIR   : /home/x232886/PycharmProjects/PV080_biometrics
SCANS_DIR  : /home/x232886/PycharmProjects/PV080_biometrics/images
